# Face Filter App: Traditional Computer Vision

This notebook builds a face filter using **traditional** methods only

**Pipeline for each frame**

1. Capture a frame from the webcam (`cv2.VideoCapture`) or upload a photo
2. Convert to greyscale and equalise the histogram
3. Detect faces and eyes with a **Viola-Jones Haar cascade** (`cv2.CascadeClassifier`)
4. Apply the filter 
5. Display the result


**Requirements:** OpenCV Contrib 4.x, NumPy and Matplotlib. Run this notebook locally so it can reach your webcam. 
OpenCV Contrib is an extension of the OpenCV (Open Source Computer Vision) library in Python. It extends the capabilities of the OpenCV library in Python by offering additional features and modules. These extras aren't in the main OpenCV library for reasons like being experimental, serving niche needs, or relying on third-party tools. With these added modules, you can do a lot more in computer vision, like detecting features, tweaking images, tracking objects, diving into machine learning, and much more.


In [127]:
# Install if needed - OpenCV Contrib 4.x  this is a prerequisite for using Haar cascades
# !pip install "opencv-contrib-python<5" numpy matplotlib

In [128]:
# Shows where python is running from
#import sys
#print(sys.executable)   # the Python the kernel is actually running
#print(sys.version)      # its Python version

In [129]:
# Show a list of installed Python libraries
# !pip list
# show the library details of opencv-contrib-python
# !pip show opencv-contrib-python

# Face Filter Application



## Imports

In [130]:
import tkinter as tk
from tkinter import ttk, filedialog
from PIL import Image, ImageTk

import os
import cv2

###
# Importing face cascade
###
def import_cascades():
    CASCADE_DIR = "haarcascades/"   # local subfolder
    if not os.path.isdir(CASCADE_DIR):
        CASCADE_DIR = cv2.data.haarcascades

    face_cascade = cv2.CascadeClassifier(CASCADE_DIR+"haarcascade_frontalface_default.xml")
    eye_cascade = cv2.CascadeClassifier(CASCADE_DIR+"haarcascade_eye.xml")

    print("Face cascade loaded:", not face_cascade.empty())
    print("Eye cascade loaded: ", not eye_cascade.empty())

    return face_cascade, eye_cascade

## File Handling

### Video

In [131]:
def update_video():

    if cap is None:
        return

    ret, frame = cap.read()

    if not ret:
        cap.release()
        return

    faces = detect_faces(frame)

    current_filter = filter_option.get()

    frame = apply_filter(current_filter, frame, faces)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)

    imgtk = ImageTk.PhotoImage(image=img)

    display_label.imgtk = imgtk
    display_label.config(image=imgtk)

    root.after(15, update_video)

def open_video():

    global cap
    if cap is not None:
        cap.release()

    path = filedialog.askopenfilename(
        filetypes=[
            ("Video Files", "*.mp4 *.avi *.webm *.mov"),
        ]
    )

    if not path:
        return

    cap = cv2.VideoCapture(path)

    update_video()


## Image

In [132]:
def update_image():
    if current_image is None:
        return

    image = current_image.copy()

    faces = detect_faces(image)

    current_filter = filter_option.get()

    image = apply_filter(current_filter, image, faces)

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img = Image.fromarray(image_rgb)
    img.thumbnail((900, 600))

    imgtk = ImageTk.PhotoImage(img)

    display_label.imgtk = imgtk
    display_label.config(image=imgtk)


def open_image():

    global current_image

    path = filedialog.askopenfilename(
        filetypes=[
            ("Image Files", "*.jpg *.png *.jpeg *.webm *.bmp"),
        ]
    )

    if not path:
        return

    current_image = cv2.imread(path)

    update_image()

## Webcam

In [133]:
def start_webcam():
    global cap

    if cap is not None:
        cap.release()

    cap = cv2.VideoCapture(0)

    update_video()

def stop_video():
    global cap

    if cap:
        cap.release()
        cap = None

    current_image = None

    display_label.config(
        image="",
        text="No media loaded"
    )

    display_label.imgtk = None

## Detection & Tracking

### Face Detection


In [134]:
def detect_faces(frame):
    # sets the frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    #detects the coordinates of the faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    return faces

## Filters

### Applying Filter

In [135]:
def apply_filter(filter_choice, frame, faces):
    #matches user choice to equivalent filter
    match filter_choice:
        case "None":
            return frame
        case "Blur Face":
            frame = blur_face(frame, faces)
        case "Gray Face":
            frame = grayscale_face(frame, faces)
        case _:
            print("Invalid filter: No filter Applied")

    return frame

def on_filter_change(event=None):

    if current_image is not None:
        update_image()

### Filters

In [136]:
def blur_face(frame, faces):

    for (x, y, w, h) in faces:
       #gets region of interest of face
        roi = frame[y:y+h, x:x+w]

        #adds blur
        blurred_face = cv2.GaussianBlur(roi, (99, 99), 25)

        #merges roi with original frame
        frame[y:y+h, x:x+w] = blurred_face

    return frame

def grayscale_face(frame, faces):
    for (x, y, w, h) in faces:
       #gets region of interest of face
        roi = frame[y:y+h, x:x+w]

        #converts face to single channel grayscale
        grayscale_face = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        #converts back to 3 channels but it is still gray
        grayscale_face = cv2.cvtColor(grayscale_face, cv2.COLOR_GRAY2BGR)

        #merges roi with original frame
        frame[y:y+h, x:x+w] = grayscale_face

    return frame

## Menu

### Filter Dropbox

In [137]:
def filter_choice_menu():
    global filter_option
    # Sets default value to no filter
    filter_option = tk.StringVar(value="None")

    # Drop down box that selects filter
    filter_menu = ttk.Combobox(
        root,
        textvariable=filter_option,
        values=[
            "None",
            "Blur Face",
            "Gray Face"
        ],
        # Can only read and select the filters
        state="readonly"
    )
    filter_menu.pack()

    filter_menu.bind("<<ComboboxSelected>>", on_filter_change
    )

### Window Setup

In [138]:
cap = None
filter_option = None
current_image = None
face_cascade, eye_cascade = import_cascades()

# Sets up the main display window
root = tk.Tk()
root.title("Face Filter System")
root.geometry("1100x800")

# Title
title_label = tk.Label(
    root,
    text="Face Filter System",
    font=("Arial", 20, "bold")
)
title_label.pack(pady=10)

# Filter Selection
filter_frame = tk.Frame(root)
filter_frame.pack(pady=5)

filter_label = tk.Label(
    filter_frame,
    text="Filter:",
)
filter_label.pack(side="left", padx=5)

filter_choice_menu()
# Toolbar for buttons
toolbar = tk.Frame(root)
toolbar.pack(pady=10)

# Display Area
display_frame = tk.Frame(
    root,
    width=900,
    height=600,
    relief="solid",
    borderwidth=2,
)

display_frame.pack(pady=20)
display_frame.pack_propagate(False)

display_label = tk.Label(
    display_frame,
    text="No media loaded",
    font=("Arial", 14)
)

display_label.pack(expand=True)

# Buttons
video_button = tk.Button(
        toolbar,
        text="Open Video",
        width=15,
        command=open_video
    )

image_button = tk.Button(
        toolbar,
        text="Open Image",
        width=15,
        command=open_image
    )

webcam_button = tk.Button(
        toolbar,
        text="Start Webcam",
        width="15",
        command=start_webcam
    )

stop_button = tk.Button(
        toolbar,
        text="Stop",
        width=15,
        command=stop_video
    )
video_button.pack(side="left", padx=5)
image_button.pack(side="left", padx=5)
webcam_button.pack(side="left", padx=5)
stop_button.pack(side="left", padx=5)

root.mainloop()


Face cascade loaded: True
Eye cascade loaded:  True
